# Peramalan Inflasi Indonesia Berdasarkan IHK
## Hybrid SARIMAX–LSTM dengan Indikator Makroekonomi Domestik dan Global

**Penulis:** Attala Omar Kareem  
**Program Studi:** S1 Teknologi Sains Data, Universitas Airlangga  
**Periode:** Januari 1990–Desember 2025  
**Target:** Inflasi IHK Indonesia *Year-on-Year* (YoY)

Notebook ini merupakan versi publik dari kode yang digunakan untuk menghasilkan keluaran akhir pada folder `results/`.


### Cara Menjalankan

1. Instal dependensi melalui `pip install -r requirements.txt`.
2. Buka notebook dari folder `notebooks/`.
3. Jalankan seluruh sel secara berurutan.
4. Data Excel sumber dibaca dari `data/raw/`.
5. Keluaran baru disimpan pada `results/final_run/`.

Keluaran hasil revisi yang sudah dipilih tersedia pada folder `results/tables/` dan `results/figures/`.


## Ringkasan 12 Skenario (Peta Indikator)

| Skenario | Basis | Indikator Domestik | Indikator Global |
|---|---|---|---|
| SARIMA_BASE | SARIMAX (exog=None) | - | - |
| SARIMAX_DOM | SARIMAX | Suku Bunga, Kurs, Cad. Devisa | - |
| SARIMAX_GLOBAL | SARIMAX | - | Minyak, Pangan, Emas, VIX |
| SARIMAX_ALL | SARIMAX | Suku Bunga, Kurs, Cad. Devisa | Minyak, Pangan, Emas, VIX |
| LSTM_BASE | LSTM univariat | - | - |
| LSTM_DOM | LSTM | Suku Bunga, Kurs, Cad. Devisa | - |
| LSTM_GLOBAL | LSTM | - | Minyak, Pangan, Emas, VIX |
| LSTM_ALL | LSTM | Suku Bunga, Kurs, Cad. Devisa | Minyak, Pangan, Emas, VIX |
| Hybrid_BASE | SARIMA_BASE + LSTM(residual) | - | - |
| Hybrid_DOM | SARIMAX_DOM + LSTM(residual) | Suku Bunga, Kurs, Cad. Devisa | - |
| Hybrid_GLOBAL | SARIMAX_GLOBAL + LSTM(residual) | - | Minyak, Pangan, Emas, VIX |
| Hybrid_ALL | SARIMAX_ALL + LSTM(residual) | Suku Bunga, Kurs, Cad. Devisa | Minyak, Pangan, Emas, VIX |

**Target (Y):** Inflasi IHK YoY (%) — Persamaan 2.2: `Y_t = (IHK_t - IHK_(t-12)) / IHK_(t-12) x 100%`
**Periode:** Januari 1990 - Desember 2025.

> Catatan: tabel ini teks statis untuk pembaca. Definisi sesungguhnya ada di variabel `SARIMAX_SCENARIOS` / `LSTM_SCENARIOS` / `HYBRID_SCENARIOS` pada cell konfigurasi di bawah. Kalau kolom indikator diubah di sana, sinkronkan juga tabel ini.


## 0. Setup: Install & Import Library

In [ ]:
# Instalasi opsional
# !pip install -r ../requirements.txt

import itertools
import os
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", ConvergenceWarning)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.grid": True,
    "grid.alpha": 0.30,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## 1. Konfigurasi Path & Konstanta

In [ ]:
# Lokasi data dan keluaran berbasis struktur repositori
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR = str(PROJECT_ROOT / "data" / "raw") + os.sep
OUTPUT_DIR = str(PROJECT_ROOT / "results" / "final_run") + os.sep
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mode tampilan
PRESENTATION_MODE = True
SHOW_EDA_INDIVIDUAL = False
SHOW_SARIMAX_DIAGNOSTICS = False
SHOW_TUNING_TOP10 = False
SHOW_TRAINING_CURVES = False
SHOW_FORECAST_INDIVIDUAL = False
SHOW_HYBRID_RESIDUAL_PLOTS = False
SHOW_EXPORT_FILE_LIST = False

# Pembagian data
PROP_TRAIN_S1 = 0.80
PROP_VAL_S1 = 0.10

# Konfigurasi SARIMAX
SEASONAL_PERIOD = 12
LB_LAGS = [12, 20, 24]
LB_ALPHA = 0.05

# Konfigurasi LSTM
LOOKBACK = 12
MAX_EPOCH = 200
PATIENCE = 20

LSTM_GRID = {
    "units": [32, 64, 128],
    "dropout": [0.1, 0.2, 0.3],
    "batch_size": [16, 32],
    "learning_rate": [1e-3, 5e-4, 1e-4],
}

# Variabel penelitian
TARGET = "Inflasi_YoY"
COLS_BASE = []
COLS_DOM = ["dSukuBunga", "dlog_Kurs", "dlog_CadDev"]
COLS_GLOBAL = ["dlog_Minyak", "dlog_Pangan", "dlog_Emas", "dVIX"]
COLS_ALL = COLS_DOM + COLS_GLOBAL

# Dua belas skenario pemodelan
SARIMAX_SCENARIOS = {
    "SARIMA_BASE": None,
    "SARIMAX_DOM": COLS_DOM,
    "SARIMAX_GLOBAL": COLS_GLOBAL,
    "SARIMAX_ALL": COLS_ALL,
}

LSTM_SCENARIOS = {
    "LSTM_BASE": COLS_BASE,
    "LSTM_DOM": COLS_DOM,
    "LSTM_GLOBAL": COLS_GLOBAL,
    "LSTM_ALL": COLS_ALL,
}

HYBRID_SCENARIOS = {
    "Hybrid_BASE": (COLS_BASE, "SARIMA_BASE"),
    "Hybrid_DOM": (COLS_DOM, "SARIMAX_DOM"),
    "Hybrid_GLOBAL": (COLS_GLOBAL, "SARIMAX_GLOBAL"),
    "Hybrid_ALL": (COLS_ALL, "SARIMAX_ALL"),
}

MODEL_ORDER = (
    list(SARIMAX_SCENARIOS)
    + list(LSTM_SCENARIOS)
    + list(HYBRID_SCENARIOS)
)

config_summary = pd.DataFrame({
    "Kelompok": ["Domestik", "Global", "Seluruh model"],
    "Jumlah": [len(COLS_DOM), len(COLS_GLOBAL), len(MODEL_ORDER)],
    "Isi": [", ".join(COLS_DOM), ", ".join(COLS_GLOBAL), "12 skenario"],
})
display(config_summary)


## 2. Load & Merge Data

In [ ]:
def load_xlsx(fname, date_col, val_col, new_name):
    df = pd.read_excel(os.path.join(DATA_DIR, fname), parse_dates=[date_col])
    df = df[[date_col, val_col]].rename(columns={date_col:'Date', val_col:new_name})
    df['Date'] = pd.to_datetime(df['Date']).dt.to_period('M').dt.to_timestamp('s')
    return df.set_index('Date').sort_index()

inflasi    = load_xlsx('inflasi_ihk_yoy_source.xlsx',   'Tanggal', 'Inflasi',             'Inflasi_YoY')
suku_bunga = load_xlsx('suku_bunga_acuan_source.xlsx',          'Date',    'Suku_Bunga',           'SukuBunga')
kurs       = load_xlsx('kurs_usd_idr_source.xlsx',    'Date',    'Terakhir',             'Kurs_USDIDR')
caddev     = load_xlsx('cadangan_devisa_source.xlsx',          'Date',    'Cadangan_Devisa(USD)', 'CadDev')
minyak     = load_xlsx('harga_minyak_wti_source.xlsx',       'Date',    'Harga_Minyak',        'Minyak_WTI')
pangan     = load_xlsx('harga_pangan_global_source.xlsx',       'Date',    'Harga_Pangan',        'Pangan')
emas       = load_xlsx('harga_emas_dunia_source.xlsx',         'Date',    'Harga_Emas(USD)',     'Emas')
vix        = load_xlsx('indeks_vix_source.xlsx', 'Date',    'Indeks_Global',       'VIX')

raw = inflasi
for d in [suku_bunga, kurs, caddev, minyak, pangan, emas, vix]:
    raw = raw.join(d, how='inner')

raw = raw.loc['1990-01':'2025-12'].copy()
raw.index.freq = 'MS'

# Tabel 4.1 Ringkasan Data
desc_rows = [
    ('Inflasi IHK YoY',   'Target',   'BPS, BI, FRED',          len(inflasi)),
    ('Suku Bunga Acuan',  'Domestik', 'BI, FRED',                len(suku_bunga)),
    ('Kurs USD/IDR',      'Domestik', 'FRED',                    len(kurs)),
    ('Cadangan Devisa',   'Domestik', 'FRED',                    len(caddev)),
    ('Harga Minyak Dunia','Global',   'EIA',                     len(minyak)),
    ('Harga Pangan Dunia','Global',   'FRED (Beras Thailand)',   len(pangan)),
    ('Harga Emas Dunia',  'Global',   'World Gold Council',      len(emas)),
    ('Indeks VIX',        'Global',   'FRED',                    len(vix)),
]
df_data_summary = pd.DataFrame(desc_rows, columns=['Variabel','Kategori','Sumber','Obs'])
df_data_summary.to_csv(OUTPUT_DIR + 'tabel_4_1_ringkasan_data.csv', index=False)

print('=== Tabel 4.1 Ringkasan Data ===')
display(df_data_summary)
print(f'\nDataset final: {raw.shape[0]} observasi | '
      f'{raw.index[0].strftime("%Y-%m")} — {raw.index[-1].strftime("%Y-%m")}')
print(f'Missing values: {raw.isnull().sum().sum()}')


## 3. Exploratory Data Analysis (EDA)

### 3.0 Tren Inflasi Indonesia Tahun 2018–2024

Sebelum EDA per variabel, bagian ini menyorot dinamika inflasi IHK Indonesia pada rentang 2018–2024, sejalan dengan konteks yang dibahas pada Bab I (Gambar 1.1) mengenai volatilitas inflasi pascapandemi dan lonjakan akibat guncangan harga energi global 2022. Titik puncak/terendah dan periode guncangan dianotasikan langsung dari data (bukan angka asumsi).

In [ ]:
# 3.0 Tren Inflasi Indonesia 2018-2024 (Time Series Plot Sederhana)
start_trend, end_trend = '2018-01', '2024-12'
trend = raw.loc[start_trend:end_trend, 'Inflasi_YoY']

if trend.empty:
    print(f'PERINGATAN: tidak ada data pada rentang {start_trend} - {end_trend}.')
else:
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(trend.index, trend.values, color='#D32F2F', linewidth=1.8)
    ax.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.6)

    ax.set_title('Tren Inflasi IHK Indonesia (Year-on-Year), 2018-2024',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Inflasi YoY (%)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Periode', fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', labelrotation=30, labelsize=9)
    ax.tick_params(axis='y', labelsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + 'gambar_1_1_tren_inflasi_2018_2024.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Rata-rata inflasi 2018-2024 : {trend.mean():.2f}%  |  Std : {trend.std():.2f}%')
    print(f'Rentang                     : {trend.min():.2f}% ({trend.idxmin().strftime("%b %Y")}) - '
          f'{trend.max():.2f}% ({trend.idxmax().strftime("%b %Y")})')


In [ ]:
VAR_LABELS = {
    "Inflasi_YoY": "Inflasi IHK YoY (%)",
    "SukuBunga": "Suku Bunga Acuan (%)",
    "Kurs_USDIDR": "Kurs USD/IDR",
    "CadDev": "Cadangan Devisa",
    "Minyak_WTI": "Harga Minyak Dunia",
    "Pangan": "Harga Pangan Global",
    "Emas": "Harga Emas Dunia",
    "VIX": "Volatilitas Dunia (VIX)",
}

assert set(VAR_LABELS) == set(raw.columns), "Nama variabel EDA tidak sesuai dengan dataset."

# Simpan grafik individual tanpa memenuhi notebook dengan banyak keluaran
for index, column in enumerate(raw.columns, start=1):
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(raw.index, raw[column], linewidth=1.1)
    ax.set_title(f"{VAR_LABELS[column]} (1990–2025)")
    ax.set_xlabel("Periode")
    ax.set_ylabel(VAR_LABELS[column])
    fig.tight_layout()

    filename = f"gambar_4_1_{index}_eda_ts_{column.lower()}.png"
    fig.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150, bbox_inches="tight")

    if SHOW_EDA_INDIVIDUAL:
        plt.show()
    else:
        plt.close(fig)

# Tampilkan satu ringkasan EDA yang ringkas
fig, axes = plt.subplots(4, 2, figsize=(16, 15))
for ax, column in zip(axes.flat, raw.columns):
    ax.plot(raw.index, raw[column], linewidth=1.0)
    ax.set_title(VAR_LABELS[column])
    ax.set_xlabel("Periode")

fig.suptitle("Ringkasan Deret Waktu Seluruh Variabel", y=1.01)
fig.tight_layout()
fig.savefig(
    os.path.join(OUTPUT_DIR, "gambar_4_1_eda_ringkasan.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()

print(f"{len(raw.columns)} grafik individual disimpan; satu grafik ringkasan ditampilkan.")


In [ ]:
# 3.2 Statistik Deskriptif
df_desc = raw.describe().T.round(4)
df_desc.columns = ['N','Mean','Std','Min','Q1','Median','Q3','Max']
df_desc.to_csv(OUTPUT_DIR + 'tabel_4_2_statistik_deskriptif.csv')
print('=== Tabel 4.2 Statistik Deskriptif ===')
display(df_desc)


In [ ]:
# 3.3 Heatmap Korelasi
corr = raw.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.8, linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Koefisien Korelasi (r)'},
            square=True, ax=ax)

# Anotasi manual dengan kontras warna otomatis
for i in range(len(corr)):
    for j in range(len(corr)):
        if not mask[i, j]:
            val = corr.iloc[i, j]
            txt_color = 'white' if abs(val) > 0.55 else 'black'
            ax.text(j + 0.5, i + 0.5, f'{val:.2f}',
                    ha='center', va='center',
                    color=txt_color, fontsize=11, fontweight='bold')

ax.set_title('Heatmap Korelasi Pearson (1990-2025)',
              fontsize=14, fontweight='bold', pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'gambar_4_2_heatmap_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()

rank = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print('\nKorelasi terhadap Inflasi_YoY:')
for var, val in rank.items():
    bar = '█' * int(abs(val)*20)
    print(f'  {var:<20} r = {val:+.4f}  {bar}')


## 4. Preprocessing & Transformasi Variabel

### Rumus Transformasi Variabel Eksogen (dipakai di cell di bawah)

Target **tidak** ditransformasi (sudah dalam bentuk YoY %, Persamaan 2.2).

Variabel eksogen ditransformasi agar stasioner (syarat SARIMAX, Bagian 2.6.2) dan berskala stabil untuk LSTM:

- **Δlevel** (suku bunga, VIX): `ΔX_t = X_t - X_(t-1)` — dipakai untuk variabel yang sudah berbentuk tingkat/rate.
- **Δlog** (kurs, cadangan devisa, minyak, pangan, emas): `Δlog(X_t) = log(X_t) - log(X_(t-1))` — dipakai untuk variabel harga/nilai level, setara pertumbuhan persentase, menstabilkan varians.

Rujukan: Bagian 3.3.2 Data Preprocessing proposal — "variabel berbentuk harga/nilai absolut ditransformasi Δlog, variabel yang sudah berbentuk tingkat ditransformasi Δlevel."


In [ ]:
# 4.1 Transformasi
# Variabel level harga/nilai → Δlog; variabel sudah dalam rate/indeks → Δlevel
data_tr = pd.DataFrame(index=raw.index)
data_tr[TARGET]       = raw['Inflasi_YoY']                  # target tidak ditransformasi
data_tr['dSukuBunga'] = raw['SukuBunga'].diff()             # Δlevel
data_tr['dlog_Kurs']  = np.log(raw['Kurs_USDIDR']).diff()   # Δlog
data_tr['dlog_CadDev']= np.log(raw['CadDev']).diff()        # Δlog
data_tr['dlog_Minyak']= np.log(raw['Minyak_WTI']).diff()    # Δlog
data_tr['dlog_Pangan']= np.log(raw['Pangan']).diff()        # Δlog
data_tr['dlog_Emas']  = np.log(raw['Emas']).diff()          # Δlog
data_tr['dVIX']       = raw['VIX'].diff()                   # Δlevel

data_tr = data_tr.dropna().copy()
data_tr.index.freq = 'MS'
N_TOTAL = len(data_tr)

print(f'Data setelah transformasi: {data_tr.shape}')
print(f'Periode: {data_tr.index[0].strftime("%Y-%m")} — {data_tr.index[-1].strftime("%Y-%m")}')
display(data_tr.head(5))


In [ ]:
# 4.2 Uji Stasioneritas ADF
adf_rows = []
for col in data_tr.columns:
    s = adfuller(data_tr[col].dropna(), autolag='AIC')
    adf_rows.append({
        'Variabel'      : col,
        'ADF Statistic' : round(s[0], 4),
        'p-value'       : round(s[1], 6),
        'Kritis 5%'     : round(s[4]['5%'], 4),
        'Lag'           : s[2],
        'Keputusan'     : 'Stasioner (✓)' if s[1] < 0.05 else 'Tidak Stasioner (✗)'
    })
df_adf = pd.DataFrame(adf_rows)
df_adf.to_csv(OUTPUT_DIR + 'tabel_4_3_uji_adf.csv', index=False)
print('=== Tabel 4.3 Hasil Uji ADF (setelah transformasi) ===')
display(df_adf)


### 4.3 Normalisasi Data

Normalisasi menggunakan **Min-Max Scaling** untuk menyamakan skala antarvariabel sebelum digunakan pada model LSTM dan *Hybrid* SARIMAX-LSTM. Parameter scaler (`min_`, `scale_`) di-fit hanya pada masing-masing **train set** di tahap pemodelan untuk mencegah *data leakage*. Tabel berikut disajikan sebagai dokumentasi rentang nilai sebelum dan sesudah transformasi.

In [ ]:
# 4.3 Normalisasi Data (MinMaxScaler)
# Fit scaler pada FULL data_tr hanya untuk dokumentasi tabel (bukan untuk model)
# Scaler aktual untuk masing-masing model di-fit pada train set masing-masing

# Label variabel yang lebih deskriptif untuk tabel
VAR_LABELS = {
    TARGET         : 'Inflasi IHK Indonesia',
    'dSukuBunga'   : 'Suku Bunga Acuan',
    'dlog_Kurs'    : 'Nilai Tukar USD/IDR',
    'dlog_CadDev'  : 'Cadangan Devisa',
    'dlog_Minyak'  : 'Harga Minyak Dunia',
    'dlog_Pangan'  : 'Harga Pangan Global',
    'dlog_Emas'    : 'Harga Emas Global',
    'dVIX'         : 'Indeks Volatilitas Global (VIX)',
}

sc_display = MinMaxScaler()
scaled_arr = sc_display.fit_transform(data_tr.values)
data_scaled = pd.DataFrame(scaled_arr, columns=data_tr.columns, index=data_tr.index)

norm_rows = []
for col in data_tr.columns:
    label = VAR_LABELS.get(col, col)
    norm_rows.append({
        'Variabel'                         : label,
        'Nilai Minimum Awal'               : round(data_tr[col].min(), 4),
        'Nilai Maksimum Awal'              : round(data_tr[col].max(), 4),
        'Nilai Minimum Setelah Normalisasi': round(data_scaled[col].min(), 4),
        'Nilai Maksimum Setelah Normalisasi': round(data_scaled[col].max(), 4),
    })

df_norm = pd.DataFrame(norm_rows)
df_norm.to_csv(OUTPUT_DIR + 'tabel_4_5_normalisasi_data.csv', index=False)

print('=== Tabel 4.5 Hasil Normalisasi Data (MinMax Scaling) ===')
print('Catatan: Scaler ini di-fit pada seluruh data_tr untuk keperluan dokumentasi.')
print('Scaler aktual model di-fit hanya pada train set (tidak terjadi data leakage).\n')
display(df_norm)


## 5. Data Splitting — Stage 1 (80/10/10) & Stage 2 (90/10)

In [ ]:
n = N_TOTAL
n_train_s1 = int(n * PROP_TRAIN_S1)          # 80%
n_val_s1   = int(n * PROP_VAL_S1)            # 10%
n_test     = n - n_train_s1 - n_val_s1       # 10%
n_train_s2 = n_train_s1 + n_val_s1           # 90%

# Stage 1
train_s1 = data_tr.iloc[:n_train_s1]
val_s1   = data_tr.iloc[n_train_s1 : n_train_s2]
test     = data_tr.iloc[n_train_s2:]

y_train_s1 = train_s1[TARGET]
y_val_s1   = val_s1[TARGET]
y_test     = test[TARGET]

# Stage 2
train_s2 = data_tr.iloc[:n_train_s2]          # 90%
y_train_s2 = train_s2[TARGET]
# test tetap sama

# Tabel split
split_info = pd.DataFrame({
    'Subset'    : ['Train S1','Val S1','Test (Holdout)','Train S2 (final)','Test (Holdout)'],
    'Tahap'     : ['Stage 1','Stage 1','Stage 1 & 2','Stage 2','Stage 2'],
    'Periode'   : [
        f'{train_s1.index[0].strftime("%b %Y")} — {train_s1.index[-1].strftime("%b %Y")}',
        f'{val_s1.index[0].strftime("%b %Y")} — {val_s1.index[-1].strftime("%b %Y")}',
        f'{test.index[0].strftime("%b %Y")} — {test.index[-1].strftime("%b %Y")}',
        f'{train_s2.index[0].strftime("%b %Y")} — {train_s2.index[-1].strftime("%b %Y")}',
        f'{test.index[0].strftime("%b %Y")} — {test.index[-1].strftime("%b %Y")}',
    ],
    'N'       : [n_train_s1, n_val_s1, n_test, n_train_s2, n_test],
    'Proporsi': [f'{n_train_s1/n*100:.1f}%', f'{n_val_s1/n*100:.1f}%',
                 f'{n_test/n*100:.1f}%', f'{n_train_s2/n*100:.1f}%',
                 f'{n_test/n*100:.1f}%'],
})
split_info.to_csv(OUTPUT_DIR + 'tabel_4_4_data_splitting.csv', index=False)
print('=== Tabel 4.4 Pembagian Data ===')
display(split_info)


In [ ]:
# Visualisasi Pembagian Data
fig, axes = plt.subplots(2, 1, figsize=(15, 7))

for ax, title, tr, vl, te, s1_lbl, s2_lbl in [
    (axes[0],
     'Stage 1 — Holdout (80 / 10 / 10): Pencarian Hyperparameter',
     train_s1, val_s1, test, 'Train 80%', 'Val 10%'),
    (axes[1],
     'Stage 2 — Final Evaluation (90 / 10): Training Final + Evaluasi',
     train_s2, None, test, 'Train 90%', None),
]:
    ax.plot(tr.index, tr[TARGET], color='#2196F3', lw=1.2, label=s1_lbl)
    if vl is not None:
        ax.plot(vl.index, vl[TARGET], color='#FF9800', lw=1.2, label=s2_lbl)
    ax.plot(te.index, te[TARGET], color='#F44336', lw=1.8, label='Test 10%')
    ax.axvline(x=train_s2.index[-1], color='#333', ls='--', lw=1)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel('Inflasi YoY (%)')
    ax.legend(fontsize=9)

plt.suptitle('Skema Pembagian Data (Stage 1 & Stage 2)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'gambar_4_3_data_splitting.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Fungsi Evaluasi (MAE, RMSE, MASE, Ljung-Box)

## Rumus Metrik Evaluasi (Persamaan 2.27-2.29)

- **MAE** = (1/n) Σ |y_t - ŷ_t| — rata-rata error absolut, satuan sama dengan data asli.
- **RMSE** = sqrt( (1/n) Σ (y_t - ŷ_t)^2 ) — memberi penalti lebih besar untuk error ekstrem.
- **MASE** = MAE_model / MAE_naive, dengan MAE_naive dihitung dari selisih 1-lag pada **data latih** (m=1). MASE < 1 → model lebih baik dari naive forecast.

Fungsi `compute_mase` dan `evaluate_model` di bawah ini adalah implementasi langsung dari ketiga rumus tsb.


In [ ]:
def compute_mase(y_true, y_pred, y_train_arr, m=1):
    mae_m  = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
    naive  = np.mean(np.abs(np.diff(np.asarray(y_train_arr), n=m)))
    return mae_m / naive if naive != 0 else np.nan

def compute_mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = np.abs(y_true) > 0.01          # abaikan nilai aktual mendekati nol
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate_model(y_true, y_pred, y_train_arr, label=''):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mase = compute_mase(y_true, y_pred, y_train_arr)
    mape = compute_mape(y_true, y_pred)
    if label:
        print(f'  {label:<40}  MAE={mae:.4f}  RMSE={rmse:.4f}  '
              f'MASE={mase:.4f}  MAPE={mape:.2f}%')
    return {'MAE': mae, 'RMSE': rmse, 'MASE': mase, 'MAPE': mape}

def ljung_box_table(resid, lags=LB_LAGS, label=''):
    rows = []
    for lag in lags:
        lb = acorr_ljungbox(resid, lags=[lag], return_df=True)
        p  = float(lb['lb_pvalue'].values[0])
        q  = float(lb['lb_stat'].values[0])
        rows.append({'Lag': lag, 'Q-Stat': round(q,4), 'p-value': round(p,4),
                     'White Noise': 'Ya (✓)' if p > LB_ALPHA else 'Tidak (✗)'})
    df = pd.DataFrame(rows)
    if label:
        print(f'  Ljung-Box [{label}]:')
        print(df.to_string(index=False))
    is_wn = all(r['p-value'] > LB_ALPHA for r in rows)
    return df, is_wn

print('Fungsi evaluasi siap.')


## 7. Analisis ACF / PACF — Seri Target (Training S1)

In [ ]:
# ACF/PACF digunakan untuk mengidentifikasi kandidat orde SARIMAX sebelum fitting
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
plot_acf( y_train_s1, lags=36, ax=axes[0], alpha=0.05)
axes[0].set_title('ACF — Inflasi YoY (Training Set S1)', fontweight='bold')
axes[0].set_xlabel('Lag (bulan)')
axes[0].axvline(x=12, color='red', ls=':', alpha=0.6, label='Lag-12')
axes[0].legend(fontsize=8)
plot_pacf(y_train_s1, lags=36, ax=axes[1], alpha=0.05, method='ywm')
axes[1].set_title('PACF — Inflasi YoY (Training Set S1)', fontweight='bold')
axes[1].set_xlabel('Lag (bulan)')
axes[1].axvline(x=12, color='red', ls=':', alpha=0.6, label='Lag-12')
axes[1].legend(fontsize=8)
plt.suptitle('ACF dan PACF Inflasi YoY (Sebelum Fitting SARIMAX)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'gambar_4_4_acf_pacf_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('Petunjuk: cut-off PACF → orde AR (p); cut-off ACF → orde MA (q)')
print('Spike di lag-12 → kandidat komponen musiman (P, Q)')


## 8. Fungsi SARIMAX (Grid Search + Ljung-Box + Rolling Forecast + Diagnostik)

## Rumus & Prosedur SARIMAX (Persamaan 2.15-2.16, Bagian 2.7.3)

Bentuk umum: `Φ_P(B^s) φ_p(B) (1-B)^d (1-B^s)^D (y_t - c - β^T x_t) = Θ_Q(B^s) θ_q(B) ε_t`

Prosedur Box-Jenkins yang diimplementasikan fungsi `select_sarimax_order` di bawah:
1. **Uji ADF** → tentukan orde differencing `d` (Persamaan 2.9-2.10).
2. **Grid search** kombinasi (p, q, P, Q) — periode musiman `s=12` ditetapkan tetap (bukan digrid), sesuai keputusan metodologis Bagian 2.10.
3. Tiap kandidat: hitung **AIC** dan uji **Ljung-Box** (Persamaan 2.30) pada residual di lag 12/20/24.
4. **Pilih**: AIC terkecil di antara kandidat yang lolos Ljung-Box (residual white noise). Fallback ke AIC terkecil murni kalau tidak ada yang lolos — status ini dicatat, bukan disembunyikan.


### 8.1 Pemilihan Orde SARIMAX

In [ ]:
def select_sarimax_order(y_train, exog_train=None, max_p=2, max_q=2,
                          s=SEASONAL_PERIOD, lb_lags=LB_LAGS, verbose=True):
    """
    Alur:
      1. ADF → d
      2. Grid search semua (p,q,P,Q)
      3. Setiap model: hitung AIC, BIC, uji Ljung-Box residual
      4. Pilih: AIC terkecil + white noise di semua lag
         Fallback: AIC terkecil jika tidak ada yang lolos Ljung-Box
    Return: order, sorder, df_top5 (DataFrame top-5 AIC)
    """
    p_val = adfuller(y_train, autolag='AIC')[1]
    d = 0 if p_val < 0.05 else 1
    exog_arr = exog_train.values if exog_train is not None else None

    combos = [(p, q, P, Q)
              for p in range(max_p+1) for q in range(max_q+1)
              for P in [0, 1] for Q in [0, 1]
              if not (p == 0 and q == 0)]

    if verbose:
        print(f'  ADF p={p_val:.4f} → d={d} | {len(combos)} kandidat...')

    records = []
    for p, q, P, Q in combos:
        try:
            order  = (p, d, q)
            sorder = (P, 0, Q, s)
            mod = SARIMAX(y_train, exog=exog_arr, order=order, seasonal_order=sorder,
                          enforce_stationarity=False, enforce_invertibility=False)
            res = mod.fit(disp=False, maxiter=200)
            lb_ps = []
            for lag in lb_lags:
                lb = acorr_ljungbox(res.resid, lags=[lag], return_df=True)
                lb_ps.append(float(lb['lb_pvalue'].values[0]))
            lb_ok  = all(pp > LB_ALPHA for pp in lb_ps)
            records.append({
                'order': order, 'sorder': sorder,
                'AIC': round(res.aic, 2), 'BIC': round(res.bic, 2),
                'LB_min_p': round(min(lb_ps), 4),
                'lb_ok': lb_ok, 'result': res
            })
        except Exception:
            pass

    if not records:
        print('  WARN: tidak ada model berhasil. Default (1,d,1)(0,0,0,s).')
        return (1, d, 1), (0, 0, 0, s), pd.DataFrame()

    records.sort(key=lambda x: x['AIC'])
    wn = [r for r in records if r['lb_ok']]
    best = wn[0] if wn else records[0]

    df_top5 = pd.DataFrame([{
        'Order SARIMAX': f"{r['order']}{r['sorder']}",
        'AIC': r['AIC'], 'BIC': r['BIC'],
        'LB min p': r['LB_min_p'],
        'White Noise': 'Ya (✓)' if r['lb_ok'] else 'Tidak (✗)',
        'Terpilih': '◀ BEST' if r is best else ''
    } for r in records[:5]])

    if verbose:
        lb_status = 'White Noise ✓' if best['lb_ok'] else 'Non-WN (terpaksa, AIC terbaik)'
        print(f'  Terpilih: order={best["order"]} sorder={best["sorder"]} '
              f'AIC={best["AIC"]} | {lb_status}')
        print()
        print(df_top5.to_string(index=False))
    return best['order'], best['sorder'], df_top5


### 8.2 Prediksi *Rolling One-Step-Ahead*

In [ ]:
def sarimax_rolling_s2(y_train_s2, exog_tr_s2, order, sorder, y_test, exog_te):
    """
    Stage 2: Rolling 1-step-ahead SARIMAX dengan expanding window.
    Fit awal pada train_s2, warm-start di setiap langkah berikutnya.
    Return: preds, resid_train (dari fit awal), result_initial
    """
    has_exog = exog_tr_s2 is not None
    exog_s2_arr = exog_tr_s2.values if has_exog else None

    # Fit awal pada train_s2
    mod0 = SARIMAX(list(y_train_s2), exog=exog_s2_arr, order=order,
                   seasonal_order=sorder, enforce_stationarity=False,
                   enforce_invertibility=False)
    res0 = mod0.fit(disp=False, maxiter=300)
    init_params = res0.params
    resid_train = res0.resid

    # Rolling
    hist_y    = list(y_train_s2)
    hist_exog = list(exog_s2_arr) if has_exog else None
    preds = []
    print(f'    Rolling ({len(y_test)} langkah)...', end=' ', flush=True)

    for i in range(len(y_test)):
        if (i+1) % 10 == 0:
            print(i+1, end=' ', flush=True)
        try:
            mod_i = SARIMAX(hist_y,
                            exog=np.array(hist_exog) if has_exog else None,
                            order=order, seasonal_order=sorder,
                            enforce_stationarity=False, enforce_invertibility=False)
            res_i = mod_i.fit(disp=False, start_params=init_params, maxiter=100)
            exog_fc = exog_te.iloc[i:i+1].values if has_exog else None
            fc = res_i.forecast(steps=1, exog=exog_fc)
            preds.append(float(np.asarray(fc).flat[0]))
            init_params = res_i.params
        except Exception as e:
            print(f"    WARN step {i}: {e}")
            preds.append(float(hist_y[-1]))
        hist_y.append(float(y_test.iloc[i]))
        if has_exog:
            hist_exog.append(exog_te.iloc[i].values)

    print('selesai.')
    return np.array(preds), resid_train, res0


### 8.3 Diagnostik Residual

In [ ]:
def diagnose_residual(resid, label, fig_num, output_dir=OUTPUT_DIR):
    """Plot ACF/PACF residual + Ljung-Box. Return (df_lb, is_wn)."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf( resid, lags=30, ax=axes[0], alpha=0.05)
    axes[0].set_title(f'ACF Residual — {label}', fontweight='bold')
    axes[0].set_xlabel('Lag')
    plot_pacf(resid, lags=30, ax=axes[1], alpha=0.05, method='ywm')
    axes[1].set_title(f'PACF Residual — {label}', fontweight='bold')
    axes[1].set_xlabel('Lag')
    plt.suptitle(f'Diagnostik Residual SARIMAX: {label}',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    fname = f'gambar_{fig_num}_residual_{label.lower().replace(" ","_")}.png'
    plt.savefig(output_dir + fname, dpi=150, bbox_inches='tight')
    if SHOW_SARIMAX_DIAGNOSTICS:
        plt.show()
    else:
        plt.close(fig)
    df_lb, is_wn = ljung_box_table(resid, label=label)
    status = (
        "Residual white noise"
        if is_wn
        else "Residual masih mengandung autokorelasi"
    )
    print(f'  → {status}')
    return df_lb, is_wn

print("Fungsi SARIMAX siap.")


## 9. Fungsi LSTM (Sequences, Build, Tune, Train, Predict)

## Rumus LSTM (Persamaan 2.19-2.21)

Sel LSTM per Persamaan 2.19: forget gate `f_t`, input gate `i_t`, output gate `o_t`, cell state `C_t`, hidden state `h_t` — dihitung otomatis oleh layer `LSTM()` Keras, tidak ditulis manual di kode.

Formulasi peramalan (Persamaan 2.20-2.21):
- Univariat (LSTM_BASE): `π_(t+1) = f_LSTM(π_t, π_(t-1), ..., π_(t-k+1))`
- Multivariat (LSTM_DOM/GLOBAL/ALL): `π_(t+1) = f_LSTM(π_t, x_t, π_(t-1), x_(t-1), ..., π_(t-k+1), x_(t-k+1))`

`k` = `LOOKBACK` = 12 (jendela 1 tahun), ditetapkan di cell konfigurasi — bukan hasil grid search, sesuai keputusan Bagian 2.10.


### 9.1 Pembentukan Sekuens dan Arsitektur LSTM

In [ ]:
def make_sequences(X, y, lookback=LOOKBACK):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)


def build_lstm_model(n_feat, units=64, dropout=0.2, lr=1e-3, lookback=LOOKBACK):
    m = Sequential([
        LSTM(units, input_shape=(lookback, n_feat), return_sequences=False),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    m.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return m


### 9.2 Pencarian *Hyperparameter* LSTM

In [ ]:
def tune_lstm(X_tr, y_tr, X_vl, y_vl, n_feat, lookback=LOOKBACK, label='LSTM'):
    """
    Grid search 54 kombinasi hyperparameter LSTM.
    Training: X_tr/y_tr | Validation: X_vl/y_vl (untuk val_loss)
    Return: best_params dict, df_all_combos (54 baris, sorted val_loss)
    """
    Xs_tr, ys_tr = make_sequences(X_tr, y_tr, lookback)
    X_all = np.vstack([X_tr, X_vl]); y_all = np.concatenate([y_tr, y_vl])
    Xs_all, ys_all = make_sequences(X_all, y_all, lookback)
    Xs_vl = Xs_all[len(Xs_tr):]; ys_vl = ys_all[len(ys_tr):]

    combos = list(itertools.product(
        LSTM_GRID['units'], LSTM_GRID['dropout'],
        LSTM_GRID['batch_size'], LSTM_GRID['learning_rate']
    ))
    print(f'  [{label}] Tuning {len(combos)} kombinasi hyperparameter...')

    records, best_vl, best_p = [], np.inf, None
    for units, dropout, batch, lr in combos:
        tf.random.set_seed(SEED); np.random.seed(SEED)
        try:
            m  = build_lstm_model(n_feat, units, dropout, lr, lookback)
            es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            h  = m.fit(Xs_tr, ys_tr, validation_data=(Xs_vl, ys_vl),
                       epochs=100, batch_size=batch, callbacks=[es], verbose=0)
            vl = min(h.history['val_loss']); ne = len(h.history['loss'])
            records.append({'units':units,'dropout':dropout,'batch_size':batch,
                             'learning_rate':lr,'val_loss':round(vl,8),'epochs_stopped':ne})
            if vl < best_vl:
                best_vl = vl
                best_p  = {'units':units,'dropout':dropout,'batch_size':batch,'learning_rate':lr}
            del m; tf.keras.backend.clear_session()
        except Exception as e:
            records.append({'units':units,'dropout':dropout,'batch_size':batch,
                             'learning_rate':lr,'val_loss':np.nan,'epochs_stopped':0})

    df_all = pd.DataFrame(records).sort_values('val_loss').reset_index(drop=True)
    df_all.index += 1
    print(f'  Best: {best_p} | val_loss={best_vl:.8f}')
    if SHOW_TUNING_TOP10:
        display(df_all.head(10))
    return best_p, df_all


### 9.3 Pelatihan Final dan Prediksi

In [ ]:
def train_lstm_final(X_tr, y_tr, X_vl_es, y_vl_es, n_feat, best_p, lookback=LOOKBACK):
    """Training final LSTM dengan best_p. X_vl_es/y_vl_es dipakai early stopping saja."""
    tf.random.set_seed(SEED); np.random.seed(SEED)
    Xs_tr, ys_tr = make_sequences(X_tr, y_tr, lookback)
    X_all = np.vstack([X_tr, X_vl_es]); y_all = np.concatenate([y_tr, y_vl_es])
    Xs_all, ys_all = make_sequences(X_all, y_all, lookback)
    Xs_vl = Xs_all[len(Xs_tr):]; ys_vl = ys_all[len(ys_tr):]

    m  = build_lstm_model(n_feat, best_p['units'], best_p['dropout'],
                           best_p['learning_rate'], lookback)
    es = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)
    h  = m.fit(Xs_tr, ys_tr, validation_data=(Xs_vl, ys_vl),
               epochs=MAX_EPOCH, batch_size=best_p['batch_size'],
               callbacks=[es], verbose=0)
    print(f'    Training selesai: {len(h.history["loss"])} epoch | '
          f'final val_loss={min(h.history["val_loss"]):.6f}')
    return m, h


def predict_lstm_test(model, X_tr, X_vl, X_te, y_tr_sc, y_vl_sc, scaler_y, lookback=LOOKBACK):
    """Prediksi test set menggunakan seluruh history (tr+vl+te) sebagai konteks."""
    X_all = np.vstack([X_tr, X_vl, X_te])
    y_dummy = np.zeros(len(X_all))
    Xs_all, _ = make_sequences(X_all, y_dummy, lookback)
    n_tv = len(X_tr) + len(X_vl) - lookback
    Xs_te = Xs_all[n_tv:]
    y_sc  = model.predict(Xs_te, verbose=0).flatten()
    return scaler_y.inverse_transform(y_sc.reshape(-1,1)).flatten()


### 9.4 Kurva Pelatihan

In [ ]:
def plot_training_curve(history, label, fig_num, output_dir=OUTPUT_DIR):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history.history['loss'],     label='Train Loss', color='#2196F3', lw=1.5)
    ax.plot(history.history['val_loss'], label='Val Loss',   color='#FF9800', lw=1.5)
    ax.set_title(f'Training Curve LSTM: {label}', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend(); ax.set_xlim(left=0)
    plt.tight_layout()
    fname = f'gambar_{fig_num}_training_curve_{label.lower().replace(" ","_")}.png'
    plt.savefig(output_dir + fname, dpi=150, bbox_inches='tight')
    plt.show()

print("Fungsi LSTM siap.")


## 10. Penyimpanan Hasil Global

In [ ]:
# Kamus penyimpan semua hasil Stage 1 dan Stage 2
STAGE1 = {}   # hyperparameter terbaik
STAGE2 = {}   # prediksi + metrik final

print('Storage siap: STAGE1 & STAGE2')


## 11. STAGE 1 — Pencarian Hyperparameter (80/10/10)
### 11.1 SARIMAX: Grid Search Order + Validasi Ljung-Box

In [ ]:
# Menggunakan train_s1 (80%) untuk AIC grid search
# val_s1 TIDAK digunakan di sini — hanya untuk LSTM
# Hasil: best order per skenario → digunakan di Stage 2

print('='*65)
print('  STAGE 1 — SARIMAX ORDER SELECTION (4 skenario)')
print('='*65)

sarimax_specs = {}   # simpan (order, sorder, df_top5) per skenario

for lbl, exog_key in SARIMAX_SCENARIOS.items():
    print(f'\n[{lbl}]')
    exog = train_s1[exog_key] if exog_key else None
    order, sorder, df_top5 = select_sarimax_order(
        y_train_s1, exog_train=exog, max_p=2, max_q=2, s=SEASONAL_PERIOD
    )
    sarimax_specs[lbl] = {'order': order, 'sorder': sorder, 'df_top5': df_top5}
    fname = f'tabel_stage1_{lbl.lower()}_top5_aic.csv'
    df_top5.to_csv(OUTPUT_DIR + fname, index=False)
    print(f'  → Tersimpan: {fname}')

print(f'\n✓ Stage 1 SARIMAX selesai ({len(SARIMAX_SCENARIOS)} skenario: {list(SARIMAX_SCENARIOS.keys())}).')


### 11.2 LSTM: Hyperparameter Tuning (Grid Search 54 Kombinasi)

In [ ]:
print('='*65)
print('  STAGE 1 — LSTM HYPERPARAMETER TUNING (4 skenario)')
print('='*65)

for lbl, cols in LSTM_SCENARIOS.items():
    feat = [TARGET] + cols
    ket = f'{len(cols)} eksogen' if cols else 'univariat (hanya lag target)'
    print(f'\n[{lbl}] — {len(feat)} fitur input ({ket})')

    sc_X = MinMaxScaler(); sc_y = MinMaxScaler()
    X_tr = sc_X.fit_transform(train_s1[feat].values)
    X_vl = sc_X.transform(val_s1[feat].values)
    y_tr = sc_y.fit_transform(y_train_s1.values.reshape(-1,1)).flatten()
    y_vl = sc_y.transform(y_val_s1.values.reshape(-1,1)).flatten()

    best_p, grid = tune_lstm(X_tr, y_tr, X_vl, y_vl, n_feat=X_tr.shape[1], label=lbl)
    STAGE1[lbl] = {'best_params': best_p, 'grid': grid,
                   'sc_X': sc_X, 'sc_y': sc_y, 'feat': feat}
    fname = f'tabel_stage1_{lbl.lower()}_tuning_54.csv'
    grid.to_csv(OUTPUT_DIR + fname)
    print(f'  → Tersimpan: {fname}')

print(f'\n✓ Stage 1 LSTM Tuning selesai ({list(LSTM_SCENARIOS.keys())}).')


### 11.3 Hybrid: Penalaan LSTM pada Residual SARIMAX

In [ ]:
print('='*65)
print('  STAGE 1 — HYBRID: TUNING LSTM UNTUK RESIDUAL SARIMAX (4 skenario)')
print('='*65)

for lbl, (cols, sarimax_key) in HYBRID_SCENARIOS.items():
    ket = f'eksogen: {cols}' if cols else 'tanpa eksogen (murni residual)'
    print(f'\n[{lbl}] — LSTM pada residual {sarimax_key} | {ket}')

    spec = sarimax_specs[sarimax_key]
    exog_tr_arr = train_s1[cols].values if cols else None
    exog_vl_arr = val_s1[cols].values   if cols else None

    mod_h = SARIMAX(y_train_s1, exog=exog_tr_arr,
                     order=spec['order'], seasonal_order=spec['sorder'],
                     enforce_stationarity=False, enforce_invertibility=False)
    res_h = mod_h.fit(disp=False, maxiter=300)
    resid_h_tr = res_h.resid.values

    n_s1 = len(train_s1)
    pred_h_vl  = res_h.predict(start=n_s1, end=n_s1+len(val_s1)-1, exog=exog_vl_arr)
    resid_h_vl = y_val_s1.values - np.asarray(pred_h_vl)

    q5_h, q95_h = np.percentile(resid_h_tr, [5, 95])
    resid_h_tr_c = np.clip(resid_h_tr, q5_h, q95_h)
    resid_h_vl_c = np.clip(resid_h_vl, q5_h, q95_h)
    print(f'  Clipping bounds ({lbl}): [{q5_h:.4f}, {q95_h:.4f}]')

    exog_tr_feat = train_s1[cols].values if cols else np.zeros((len(train_s1), 0))
    exog_vl_feat = val_s1[cols].values   if cols else np.zeros((len(val_s1), 0))
    feat_h_tr = np.column_stack([exog_tr_feat, resid_h_tr_c])
    feat_h_vl = np.column_stack([exog_vl_feat, resid_h_vl_c])

    sc_X_h = MinMaxScaler(); sc_y_h = MinMaxScaler()
    X_h_tr = sc_X_h.fit_transform(feat_h_tr)
    X_h_vl = sc_X_h.transform(feat_h_vl)
    y_h_tr = sc_y_h.fit_transform(resid_h_tr_c.reshape(-1,1)).flatten()
    y_h_vl = sc_y_h.transform(resid_h_vl_c.reshape(-1,1)).flatten()

    best_p_h, grid_h = tune_lstm(X_h_tr, y_h_tr, X_h_vl, y_h_vl,
                                  n_feat=X_h_tr.shape[1], label=lbl)
    STAGE1[lbl] = {
        'best_params': best_p_h, 'grid': grid_h,
        'sc_X': sc_X_h, 'sc_y': sc_y_h,
        'q5': q5_h, 'q95': q95_h
    }
    fname = f'tabel_stage1_{lbl.lower()}_tuning_54.csv'
    grid_h.to_csv(OUTPUT_DIR + fname)
    print(f'  → Tersimpan: {fname}')

print(f'\n✓ Stage 1 Hybrid Tuning selesai ({list(HYBRID_SCENARIOS.keys())}).')


## 12. STAGE 2 — Training Final & Evaluasi (90/10)
### 12.1 SARIMAX Baseline, DOM, ALL — Final Training + Rolling Forecast + Diagnostik Residual

In [ ]:
print('='*65)
print('  STAGE 2 — SARIMAX FINAL TRAINING & ROLLING FORECAST (4 skenario)')
print('='*65)

FIG_RESID = {lbl: f'4.{5+i}' for i, lbl in enumerate(SARIMAX_SCENARIOS)}
all_lb_tables = {}

for lbl, exog_key in SARIMAX_SCENARIOS.items():
    print(f'\n[{lbl}] Training pada 90% data...')
    spec    = sarimax_specs[lbl]
    exog_s2 = train_s2[exog_key] if exog_key else None
    exog_te = test[exog_key]     if exog_key else None

    preds, resid_tr, res0 = sarimax_rolling_s2(
        y_train_s2, exog_s2,
        spec['order'], spec['sorder'],
        y_test, exog_te
    )

    print(f'  Diagnostik residual (training 90%):')
    df_lb, is_wn = diagnose_residual(resid_tr, lbl, FIG_RESID[lbl])
    df_lb.to_csv(OUTPUT_DIR + f'tabel_ljungbox_{lbl.lower()}.csv', index=False)
    all_lb_tables[lbl] = df_lb

    n = len(preds)
    actual = y_test.values[-n:]
    metrics = evaluate_model(actual, preds, y_train_s2.values, lbl)

    STAGE2[lbl] = {
        'pred': preds, 'actual': actual, 'metrics': metrics,
        'resid_train': resid_tr, 'order': spec['order'],
        'sorder': spec['sorder'], 'df_lb': df_lb, 'is_wn': is_wn,
        'result_s2': res0
    }
    print(f'  ✓ {lbl} selesai')

print(f'\n✓ Stage 2 SARIMAX selesai ({len(SARIMAX_SCENARIOS)} skenario).')


### 12.2 LSTM (BASE, DOM, GLOBAL, ALL) — Final Training (90%) + Prediksi Test

In [ ]:
print('='*65)
print('  STAGE 2 — LSTM FINAL TRAINING (4 skenario)')
print('='*65)

FIG_CURVE_LSTM = {lbl: f'4.{9+i}' for i, lbl in enumerate(LSTM_SCENARIOS)}

for lbl, cols in LSTM_SCENARIOS.items():
    print(f'\n[{lbl}]')
    info = STAGE1[lbl]
    feat = info['feat']
    sc_X2 = MinMaxScaler(); sc_y2 = MinMaxScaler()

    X2_tr = sc_X2.fit_transform(train_s2[feat].values)
    X2_vl = sc_X2.transform(val_s1[feat].values)     # early stopping saja
    X2_te = sc_X2.transform(test[feat].values)
    y2_tr = sc_y2.fit_transform(y_train_s2.values.reshape(-1,1)).flatten()
    y2_vl = sc_y2.transform(y_val_s1.values.reshape(-1,1)).flatten()

    print('  Training final dengan best params...')
    model_l, hist_l = train_lstm_final(
        X2_tr, y2_tr, X2_vl, y2_vl,
        n_feat=X2_tr.shape[1], best_p=info['best_params']
    )
    plot_training_curve(hist_l, lbl, FIG_CURVE_LSTM[lbl])

    pred_l = predict_lstm_test(model_l, X2_tr, X2_vl, X2_te,
                                y2_tr, y2_vl, sc_y2)
    n_l = min(len(pred_l), len(y_test))
    pred_l   = pred_l[-n_l:]
    actual_l = y_test.values[-n_l:]
    metrics_l = evaluate_model(actual_l, pred_l, y_train_s2.values, lbl)

    STAGE2[lbl] = {
        'pred': pred_l, 'actual': actual_l, 'metrics': metrics_l,
        'best_params': info['best_params'], 'history': hist_l
    }
    print(f'  ✓ {lbl} selesai')

print(f'\n✓ Stage 2 LSTM selesai ({list(LSTM_SCENARIOS.keys())}).')


### 12.3 Hybrid SARIMAX-LSTM (BASE, DOM, GLOBAL, ALL) — Final Training (90%) + Prediksi Test

## Rumus Hybrid Residual-Based (Persamaan 2.22-2.25)

1. **Residual SARIMAX**: `e_t = y_t - y_t^S` (Persamaan 2.22) — dihitung dari hasil SARIMAX basis skenario ini (`resid_train`, sudah dihasilkan di Stage 2 SARIMAX di atas).
2. **LSTM belajar pola residual**: `e_t = f(e_(t-1), ..., e_(t-k); θ)` (Persamaan 2.23).
3. **Residual clipping** (Persamaan 2.25): `e_t^clip = min(max(e_t, L), U)`, dengan `L`,`U` dari persentil 5/95 residual data latih — mencegah koreksi ekstrem.
4. **Prediksi akhir**: `y_t^(H) = y_t^(S) + e_t` (Persamaan 2.24).

**Transparansi status residual:** SARIMAX basis skenario ini TIDAK di-gate berdasarkan status white noise-nya (lihat cetakan `is_wn` di bawah) — hybrid tetap dibangun baik residual sudah WN maupun belum, karena itu bagian dari yang mau dijawab RQ4 (apakah hybrid tetap unggul walau SARIMAX-nya sudah "bersih"?).


### 12.3.1 Persiapan Data Residual Hybrid

In [ ]:
def prepare_hybrid_stage2_data(label, columns, sarimax_key):
    info = STAGE1[label]
    q5, q95 = info["q5"], info["q95"]

    residual_train = STAGE2[sarimax_key]["resid_train"]
    sarimax_test = STAGE2[sarimax_key]["pred"]
    residual_train = np.clip(residual_train, q5, q95)

    n_train_s1 = len(train_s1)
    sarimax_result = STAGE2[sarimax_key]["result_s2"]
    exog_val = val_s1[columns].values if columns else None

    sarimax_val = np.asarray(
        sarimax_result.predict(
            start=n_train_s1,
            end=n_train_s1 + len(val_s1) - 1,
            exog=exog_val,
        )
    )
    residual_val = np.clip(y_val_s1.values - sarimax_val, q5, q95)

    exog_train = (
        train_s2[columns].values
        if columns
        else np.zeros((len(train_s2), 0))
    )
    exog_val_features = (
        val_s1[columns].values
        if columns
        else np.zeros((len(val_s1), 0))
    )

    features_train = np.column_stack([exog_train, residual_train])
    features_val = np.column_stack([exog_val_features, residual_val])

    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    x_train = scaler_x.fit_transform(features_train)
    x_val = scaler_x.transform(features_val)
    y_train = scaler_y.fit_transform(residual_train.reshape(-1, 1)).ravel()
    y_val = scaler_y.transform(residual_val.reshape(-1, 1)).ravel()

    return {
        "info": info,
        "q5": q5,
        "q95": q95,
        "sarimax_test": sarimax_test,
        "residual_train": residual_train,
        "x_train": x_train,
        "x_val": x_val,
        "y_train": y_train,
        "y_val": y_val,
        "scaler_x": scaler_x,
        "scaler_y": scaler_y,
    }


### 12.3.2 Prediksi Residual pada Data Uji

In [ ]:
def predict_hybrid_residual(
    model,
    columns,
    residual_train,
    sarimax_test,
    scaler_x,
    scaler_y,
    q5,
    q95,
):
    residual_history = list(residual_train[-LOOKBACK:])
    exog_history = list(
        train_s2[columns].values[-LOOKBACK:]
        if columns
        else np.zeros((LOOKBACK, 0))
    )
    predictions = []

    for step in range(len(y_test)):
        exog_window = (
            np.asarray(exog_history[-LOOKBACK:])
            if columns
            else np.zeros((LOOKBACK, 0))
        )
        feature_window = np.column_stack([
            exog_window,
            np.asarray(residual_history[-LOOKBACK:]).reshape(-1, 1),
        ])
        feature_scaled = scaler_x.transform(feature_window)
        feature_scaled = feature_scaled.reshape(1, LOOKBACK, -1)

        prediction_scaled = model.predict(feature_scaled, verbose=0)[0, 0]
        prediction = scaler_y.inverse_transform([[prediction_scaled]])[0, 0]
        predictions.append(float(np.clip(prediction, q5, q95)))

        actual_residual = float(y_test.iloc[step]) - float(sarimax_test[step])
        residual_history.append(np.clip(actual_residual, q5, q95))

        next_exog = test[columns].iloc[step].values if columns else np.zeros(0)
        exog_history.append(next_exog)

    return np.asarray(predictions)


### 12.3.3 Pelatihan dan Evaluasi Empat Skenario Hybrid

In [ ]:
FIG_CURVE_HYB = {
    label: f"4.{13 + index}"
    for index, label in enumerate(HYBRID_SCENARIOS)
}

for label, (columns, sarimax_key) in HYBRID_SCENARIOS.items():
    white_noise = STAGE2[sarimax_key].get("is_wn")
    status = "white noise" if white_noise else "masih berautokorelasi"
    print(f"[{label}] Basis {sarimax_key}; residual {status}.")

    prepared = prepare_hybrid_stage2_data(label, columns, sarimax_key)
    model, history = train_lstm_final(
        prepared["x_train"],
        prepared["y_train"],
        prepared["x_val"],
        prepared["y_val"],
        n_feat=prepared["x_train"].shape[1],
        best_p=prepared["info"]["best_params"],
    )
    plot_training_curve(history, label, FIG_CURVE_HYB[label])

    residual_prediction = predict_hybrid_residual(
        model=model,
        columns=columns,
        residual_train=prepared["residual_train"],
        sarimax_test=prepared["sarimax_test"],
        scaler_x=prepared["scaler_x"],
        scaler_y=prepared["scaler_y"],
        q5=prepared["q5"],
        q95=prepared["q95"],
    )

    n_common = min(len(residual_prediction), len(prepared["sarimax_test"]))
    hybrid_prediction = (
        prepared["sarimax_test"][-n_common:]
        + residual_prediction[-n_common:]
    )
    actual = y_test.values[-n_common:]
    metrics = evaluate_model(
        actual,
        hybrid_prediction,
        y_train_s2.values,
        label,
    )

    STAGE2[label] = {
        "pred": hybrid_prediction,
        "actual": actual,
        "metrics": metrics,
        "best_params": prepared["info"]["best_params"],
        "history": history,
        "pred_resid_test": residual_prediction,
        "sarima_base_pred": prepared["sarimax_test"],
    }

print(f"Stage 2 Hybrid selesai: {list(HYBRID_SCENARIOS)}")


## 13. Ringkasan Hasil — Tabel untuk Bab 4

In [ ]:
# Tabel 4.5 Evaluasi 12 Model (Stage 2: 90/10)
rows_eval = []
for lbl in MODEL_ORDER:
    if lbl not in STAGE2:
        continue
    m = STAGE2[lbl]['metrics']
    rows_eval.append({'Model': lbl, 'MAE': round(m['MAE'],4),
                      'RMSE': round(m['RMSE'],4), 'MASE': round(m['MASE'],4)})
    # MAPE tersimpan di STAGE2[lbl]['metrics']['MAPE'] tapi tidak ditampilkan di tabel
df_eval = pd.DataFrame(rows_eval)
df_eval['Rank MAE'] = df_eval['MAE'].rank(ascending=True).astype(int)
df_eval.to_csv(OUTPUT_DIR + 'tabel_4_5_evaluasi_semua_model.csv', index=False)
print('=== Tabel 4.5 — Evaluasi Model (Stage 2: 90/10) ===')
display(df_eval)
print(f"\nModel terbaik (MAE): {df_eval.loc[df_eval['MAE'].idxmin(),'Model']}")

# MAPE terpisah (informatif, tidak masuk tabel utama)
print('\n--- MAPE per Model (informatif) ---')
for lbl in MODEL_ORDER:
    if lbl not in STAGE2:
        continue
    mape = STAGE2[lbl]['metrics'].get('MAPE', np.nan)
    print(f'  {lbl:<40}  MAPE={mape:.2f}%')


In [ ]:
top5_tables = []
for label in SARIMAX_SCENARIOS:
    table = sarimax_specs[label]["df_top5"].copy()
    table.insert(0, "Model", label)
    top5_tables.append(table)

df_sarimax_top5 = pd.concat(top5_tables, ignore_index=True)
display(df_sarimax_top5)


In [ ]:
lb_tables = []
for label in SARIMAX_SCENARIOS:
    if label not in STAGE2:
        continue

    table = STAGE2[label]["df_lb"].copy()
    table.insert(0, "Model", label)
    table["White Noise Keseluruhan"] = (
        "Ya" if STAGE2[label]["is_wn"] else "Tidak"
    )
    lb_tables.append(table)

df_lb_all = pd.concat(lb_tables, ignore_index=True)
df_lb_all.to_csv(
    os.path.join(OUTPUT_DIR, "tabel_4_7_ljung_box_semua.csv"),
    index=False,
)
display(df_lb_all)


In [ ]:
# Tabel 4.8 Hyperparameter LSTM & Hybrid Terbaik (Stage 1)
print('=== Tabel 4.8 — Hyperparameter Terbaik LSTM & Hybrid (Stage 1) ===')
tune_summary = []
for lbl in list(LSTM_SCENARIOS.keys()) + list(HYBRID_SCENARIOS.keys()):
    bp = STAGE1[lbl]['best_params']
    best_vl = STAGE1[lbl]['grid']['val_loss'].min()
    tune_summary.append({
        'Model': lbl, 'units': bp['units'], 'dropout': bp['dropout'],
        'batch_size': bp['batch_size'], 'learning_rate': bp['learning_rate'],
        'best_val_loss': round(best_vl, 8)
    })
df_tune = pd.DataFrame(tune_summary)
df_tune.to_csv(OUTPUT_DIR + 'tabel_4_8_best_hyperparameter_lstm.csv', index=False)
display(df_tune)


In [ ]:
for label in list(LSTM_SCENARIOS) + list(HYBRID_SCENARIOS):
    top10_table = STAGE1[label]["grid"].head(10)
    filename = f"tabel_4_9_tuning_top10_{label.lower()}.csv"
    top10_table.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)

    if SHOW_TUNING_TOP10:
        print(label)
        display(top10_table)

if not SHOW_TUNING_TOP10:
    print("Tabel Top-10 setiap skenario telah disimpan tanpa ditampilkan.")


### 13.5 Pengaruh Penambahan Variabel Eksogen (Domestik & Global) terhadap Akurasi — Menjawab RQ1

Bagian ini secara eksplisit menguji rumusan masalah pertama: **apakah penambahan indikator domestik dan global sebagai variabel eksogen membuat hasil ramalan inflasi IHK lebih baik dibandingkan model tanpa indikator tersebut?**

Pendekatan yang digunakan:
1. **Perbandingan deskriptif** — MAE, RMSE, MASE tiap skenario (`_BASE`, `_DOM`, `_GLOBAL`, `_ALL`) dalam satu keluarga model (SARIMAX, LSTM, Hybrid), dinyatakan sebagai persentase perubahan terhadap skenario `_BASE` (tanpa eksogen apa pun / hanya lag target).
2. **Uji signifikansi statistik** — Diebold-Mariano (1995) test dengan koreksi *small-sample* Harvey-Leybourne-Newbold (1997), untuk menguji apakah selisih akurasi antara model tanpa eksogen dan model dengan eksogen signifikan secara statistik, bukan sekadar kebetulan sampling pada *test set* yang kecil.

**Catatan metodologis (harus disampaikan saat sidang):**
- `SARIMA_BASE` bersifat *nested* di dalam `SARIMAX_DOM/GLOBAL/ALL` (superset kolom eksogen). Uji DM klasik dirancang untuk model *non-nested*; pada perbandingan *nested*, uji ini rentan *size distortion* (Clark & McCracken, 2001). Hasil DM pada perbandingan SARIMAX di sini diposisikan sebagai **bukti pendukung deskriptif**, bukan pembuktian inferensial yang ketat.
- Pada keluarga LSTM, tiap skenario (`BASE/DOM/GLOBAL/ALL`) memiliki hyperparameter hasil tuning terpisah (Stage 1). Artinya selisih akurasi dapat mencerminkan kombinasi *hyperparameter* terbaik yang berbeda, bukan murni efek variabel eksogen — ini adalah *confounding factor* yang perlu diakui, bukan diabaikan.
- Ukuran *test set* Stage 2 relatif kecil (±10% dari ±432 observasi ≈ 43-44 titik), sehingga uji signifikansi memiliki *power* terbatas. **Tidak signifikan secara statistik tidak sama dengan "tidak ada pengaruh"** — bisa jadi pengaruhnya ada tetapi sampel terlalu kecil untuk mendeteksinya.

In [ ]:
# Fungsi Uji Diebold-Mariano (dengan koreksi Harvey-Leybourne-Newbold)
from scipy.stats import t as _t_dist

def diebold_mariano_test(actual, pred_base, pred_alt, h=1, loss='MSE'):
    """
    Uji Diebold-Mariano (1995) dengan koreksi small-sample HLN (1997).

    H0: akurasi model 'base' dan 'alt' sama (E[d_t] = 0)
    H1: akurasi model 'alt' berbeda (lebih baik/lebih buruk) dari 'base'

    d_t = loss(e_base,t) - loss(e_alt,t)
    DM-stat > 0 & signifikan -> model 'alt' (dengan eksogen) LEBIH BAIK dari 'base'
    DM-stat < 0 & signifikan -> model 'alt' LEBIH BURUK dari 'base'

    Returns: dm_stat (teradjust HLN), p_value (two-sided, df = n-1)
    """
    actual     = np.asarray(actual, dtype=float)
    pred_base  = np.asarray(pred_base, dtype=float)
    pred_alt   = np.asarray(pred_alt, dtype=float)

    e_base = actual - pred_base
    e_alt  = actual - pred_alt

    if loss.upper() == 'MSE':
        d = e_base**2 - e_alt**2
    elif loss.upper() == 'MAE':
        d = np.abs(e_base) - np.abs(e_alt)
    else:
        raise ValueError("loss harus 'MSE' atau 'MAE'")

    n = len(d)
    d_bar = np.mean(d)

    # Varians jangka panjang dengan Newey-West (lag = h-1)
    gamma0 = np.var(d, ddof=0)
    var_d = gamma0
    for lag in range(1, h):
        gamma_k = np.cov(d[:-lag], d[lag:])[0, 1]
        var_d += 2 * gamma_k
    var_d /= n

    if var_d <= 0 or np.isnan(var_d):
        return np.nan, np.nan

    dm_stat = d_bar / np.sqrt(var_d)

    # Koreksi Harvey-Leybourne-Newbold (1997) untuk sampel kecil
    hln_factor = np.sqrt((n + 1 - 2*h + (h*(h-1))/n) / n)
    dm_stat_adj = dm_stat * hln_factor

    p_value = 2 * (1 - _t_dist.cdf(np.abs(dm_stat_adj), df=n-1))
    return dm_stat_adj, p_value

print('Fungsi diebold_mariano_test() siap.')


In [ ]:
def build_exog_effect_table(family_prefix, loss_for_dm='MSE'):
    """
    Bangun tabel perbandingan skenario BASE vs DOM vs GLOBAL vs ALL
    untuk satu keluarga model (SARIMAX / LSTM / Hybrid), lengkap dengan
    %-perubahan error terhadap BASE dan uji Diebold-Mariano vs BASE.
    """
    if family_prefix == "SARIMAX":
        order = ["SARIMA_BASE", "SARIMAX_DOM", "SARIMAX_GLOBAL", "SARIMAX_ALL"]
        base_lbl = "SARIMA_BASE"
    else:
        order = [f'{family_prefix}_BASE', f'{family_prefix}_DOM',
                 f'{family_prefix}_GLOBAL', f'{family_prefix}_ALL']
        base_lbl = f'{family_prefix}_BASE'
    labels = [label for label in order if label in STAGE2]
    if base_lbl not in STAGE2:
        print(f'  [!] {base_lbl} tidak ditemukan di STAGE2 — lewati keluarga {family_prefix}.')
        return None

    base_metrics = STAGE2[base_lbl]['metrics']
    base_actual  = STAGE2[base_lbl]['actual']
    base_pred    = STAGE2[base_lbl]['pred']

    rows = []
    for lbl in labels:
        m = STAGE2[lbl]['metrics']
        pct_mae  = (m['MAE']  - base_metrics['MAE'])  / base_metrics['MAE']  * 100
        pct_rmse = (m['RMSE'] - base_metrics['RMSE']) / base_metrics['RMSE'] * 100
        pct_mase = (m['MASE'] - base_metrics['MASE']) / base_metrics['MASE'] * 100

        row = {
            'Model': lbl,
            'MAE': round(m['MAE'], 4), 'RMSE': round(m['RMSE'], 4), 'MASE': round(m['MASE'], 4),
            '%Δ MAE vs BASE':  round(pct_mae, 2),
            '%Δ RMSE vs BASE': round(pct_rmse, 2),
            '%Δ MASE vs BASE': round(pct_mase, 2),
        }

        if lbl == base_lbl:
            row['DM-stat (vs BASE)'] = np.nan
            row['p-value']           = np.nan
            row['Signifikan (α=0.05)'] = '—'
            row['Kesimpulan']        = 'Baseline (tanpa eksogen)'
        else:
            alt_actual = STAGE2[lbl]['actual']
            alt_pred   = STAGE2[lbl]['pred']
            n_common   = min(len(base_actual), len(alt_actual))
            dm_stat, p_val = diebold_mariano_test(
                base_actual[-n_common:],
                base_pred[-n_common:],
                alt_pred[-n_common:],
                h=1, loss=loss_for_dm
            )
            sig = 'Ya' if (not np.isnan(p_val) and p_val < 0.05) else 'Tidak'
            if np.isnan(p_val):
                concl = 'Uji tidak valid (varians ≤ 0)'
            elif p_val < 0.05 and dm_stat > 0:
                concl = 'Signifikan LEBIH BAIK dari BASE'
            elif p_val < 0.05 and dm_stat < 0:
                concl = 'Signifikan LEBIH BURUK dari BASE'
            else:
                concl = 'Tidak berbeda signifikan dari BASE'

            row['DM-stat (vs BASE)']    = round(dm_stat, 4) if not np.isnan(dm_stat) else np.nan
            row['p-value']              = round(p_val, 4) if not np.isnan(p_val) else np.nan
            row['Signifikan (α=0.05)']  = sig
            row['Kesimpulan']           = concl

        rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
exog_effect_tables = {}

for family in ["SARIMAX", "LSTM", "Hybrid"]:
    table = build_exog_effect_table(family, loss_for_dm="MSE")
    if table is None:
        continue

    table.insert(0, "Keluarga", family)
    exog_effect_tables[family] = table
    table.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"tabel_4_10_pengaruh_eksogen_{family.lower()}.csv",
        ),
        index=False,
    )

df_exog_effect = pd.concat(exog_effect_tables.values(), ignore_index=True)
display(df_exog_effect)

best_exog_rows = []
for family, table in exog_effect_tables.items():
    best = table.loc[table["MAE"].idxmin()]
    best_exog_rows.append({
        "Keluarga": family,
        "Skenario Terbaik": best["Model"],
        "MAE": best["MAE"],
        "Perubahan MAE vs BASE (%)": best["%Δ MAE vs BASE"],
        "Signifikan": best["Signifikan (α=0.05)"],
    })

display(pd.DataFrame(best_exog_rows))


## 14. Visualisasi Lengkap Hasil (Gambar untuk Bab 4)

In [ ]:
# Warna & style per model (12 skenario)
COLORS = {
    'SARIMA_BASE'  : '#9E9E9E', 'SARIMAX_DOM'   : '#2196F3',
    'SARIMAX_GLOBAL': '#3F51B5', 'SARIMAX_ALL'   : '#FF9800',
    'LSTM_BASE'     : '#8BC34A', 'LSTM_DOM'      : '#4CAF50',
    'LSTM_GLOBAL'   : '#009688', 'LSTM_ALL'      : '#00BCD4',
    'Hybrid_BASE'   : '#E91E63', 'Hybrid_DOM'    : '#F44336',
    'Hybrid_GLOBAL' : '#9C27B0', 'Hybrid_ALL'    : '#673AB7',
}
STYLES = {
    'SARIMA_BASE'  : ':',  'SARIMAX_DOM'   : '-.',
    'SARIMAX_GLOBAL': '-.', 'SARIMAX_ALL'   : '--',
    'LSTM_BASE'     : ':',  'LSTM_DOM'      : '-.',
    'LSTM_GLOBAL'   : '--', 'LSTM_ALL'      : '-',
    'Hybrid_BASE'   : ':',  'Hybrid_DOM'    : '--',
    'Hybrid_GLOBAL' : '-.', 'Hybrid_ALL'    : '-',
}

# Ambil indeks test (referensi: SARIMAX_DOM selalu ada di semua skenario)
n_ref   = len(STAGE2['SARIMAX_DOM']['actual'])
idx_te  = test.index[-n_ref:]
actual  = STAGE2['SARIMAX_DOM']['actual']


### 14.1 Visualisasi Per Skenario (Individual)

Sebelum menampilkan visualisasi gabungan, berikut disajikan grafik prediksi vs aktual untuk **masing-masing dari 12 skenario model secara terpisah** (Gambar 4.11.1 – 4.11.12), agar pola deviasi tiap model dapat diamati secara individual sebelum dibandingkan secara simultan.

In [ ]:
# Gambar 4.11.1 – 4.11.12 — Prediksi vs Aktual PER SKENARIO (terpisah)
fig_codes = [f'4_11_{i+1}' for i in range(len(MODEL_ORDER))]

for i, lbl in enumerate(MODEL_ORDER):
    if lbl not in STAGE2:
        continue

    p   = STAGE2[lbl]['pred']
    n_p = len(p)
    m   = STAGE2[lbl]['metrics']

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(idx_te, actual, 'k-', lw=2.2, label='Aktual', zorder=10)
    ax.plot(idx_te[-n_p:], p, color=COLORS[lbl], linestyle=STYLES[lbl],
            lw=2, label=f'Prediksi {lbl}')
    ax.fill_between(idx_te[-n_p:], actual[-n_p:], p,
                    alpha=0.15, color=COLORS[lbl])

    ax.set_title(f"Perbandingan Nilai Aktual vs Prediksi {lbl.replace('_','-')}",
                  fontsize=13, fontweight='bold')
    ax.set_ylabel('Inflasi YoY (%)'); ax.set_xlabel('Periode')
    ax.legend(fontsize=9, loc='upper right')
    ax.text(0.01, 0.02, f"MAE={m['MAE']:.4f} | RMSE={m['RMSE']:.4f} | MASE={m['MASE']:.4f}",
            transform=ax.transAxes, fontsize=8, color='gray', va='bottom')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + f'gambar_{fig_codes[i]}_forecast_{lbl}.png',
                 dpi=150, bbox_inches='tight')
    if SHOW_FORECAST_INDIVIDUAL:
        plt.show()
    else:
        plt.close(fig)

print("Dua belas grafik prediksi individual telah disimpan.")


### 14.2 Visualisasi Gabungan (Seluruh Skenario)

Berikut visualisasi yang menggabungkan kedua belas skenario model dalam satu tampilan (overlay prediksi vs aktual, grid perbandingan individual, bar chart metrik, dan boxplot error) untuk memudahkan perbandingan langsung antar model.

In [ ]:
# Gambar 4.12 — Prediksi vs Aktual (semua model gabungan/overlay)
fig, ax = plt.subplots(figsize=(17, 7))
ax.plot(idx_te, actual, 'k-', lw=2.5, label='Aktual', zorder=10)
for lbl in MODEL_ORDER:
    if lbl not in STAGE2:
        continue
    p    = STAGE2[lbl]['pred']
    n_p  = len(p)
    mae  = STAGE2[lbl]['metrics']['MAE']
    ax.plot(idx_te[-n_p:], p,
            color=COLORS[lbl], linestyle=STYLES[lbl],
            lw=1.8, alpha=0.9, label=f'{lbl} (MAE={mae:.4f})')
ax.set_title('Perbandingan Nilai Aktual vs Prediksi 12 Skenario Model',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Inflasi YoY (%)'); ax.set_xlabel('Periode')
ax.legend(fontsize=7.5, ncol=4, loc='upper right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'gambar_4_12_forecast_semua_model.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Gambar 4.13 — Prediksi vs Aktual Individual per Model (Grid)
fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes_flat = axes.flatten()

for i, lbl in enumerate(MODEL_ORDER):
    if lbl not in STAGE2:
        continue
    ax   = axes_flat[i]
    p    = STAGE2[lbl]['pred']
    n_p  = len(p)
    m    = STAGE2[lbl]['metrics']
    ax.plot(idx_te,      actual,     'k-',  lw=2,   label='Aktual', zorder=10)
    ax.plot(idx_te[-n_p:], p, color=COLORS[lbl], lw=1.8, label='Prediksi')
    ax.fill_between(idx_te[-n_p:], actual[-n_p:], p,
                    alpha=0.15, color=COLORS[lbl])
    ax.set_title(f"Perbandingan Nilai Aktual vs Prediksi {lbl.replace('_','-')}",
                 fontsize=10, fontweight='bold')
    ax.set_ylabel('Inflasi YoY (%)'); ax.set_xlabel('Periode')
    ax.tick_params(axis='x', labelrotation=30, labelsize=7)
    ax.legend(fontsize=8)

# Sembunyikan axes kosong (jika MODEL_ORDER < jumlah slot grid)
for j in range(len(MODEL_ORDER), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Prediksi vs Aktual per Model (Stage 2)',
             fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'gambar_4_13_forecast_individual.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Gambar Tambahan — Residual Aktual vs Residual Prediksi LSTM (Hybrid)
FIG_RESID_HYB = {lbl: f'{FIG_CURVE_HYB[lbl]}b' for lbl in HYBRID_SCENARIOS}

resid_compare_rows = []

for lbl, (cols, sarimax_key) in HYBRID_SCENARIOS.items():
    pred_resid   = STAGE2[lbl]['pred_resid_test']
    sarimax_pred = STAGE2[lbl]['sarima_base_pred']
    n_h = min(len(pred_resid), len(sarimax_pred), len(y_test))

    actual_resid = y_test.values[-n_h:] - sarimax_pred[-n_h:]
    pred_resid_a = pred_resid[-n_h:]
    idx_resid    = test.index[-n_h:]

    # Metrik kecocokan residual (bukan metrik hybrid final)
    mae_resid  = np.mean(np.abs(actual_resid - pred_resid_a))
    rmse_resid = np.sqrt(np.mean((actual_resid - pred_resid_a)**2))
    corr_resid = np.corrcoef(actual_resid, pred_resid_a)[0, 1]

    resid_compare_rows.append({
        'Model': lbl, 'MAE_resid': round(mae_resid, 4),
        'RMSE_resid': round(rmse_resid, 4), 'Corr': round(corr_resid, 4)
    })

    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.axhline(0, color='gray', ls='--', lw=1, alpha=0.6)
    ax.plot(idx_resid, actual_resid, 'k-', lw=2, label='Residual Aktual (y_test - SARIMAX)', zorder=10)
    ax.plot(idx_resid, pred_resid_a, color=COLORS[lbl], lw=2, ls='--',
            label='Residual Prediksi LSTM (clipped)')
    ax.fill_between(idx_resid, actual_resid, pred_resid_a, alpha=0.15, color=COLORS[lbl])
    ax.set_title(f'Residual SARIMAX: Aktual vs Prediksi LSTM — {lbl.replace("_","-")}',
                  fontsize=12, fontweight='bold')
    ax.set_ylabel('Residual'); ax.set_xlabel('Periode')
    ax.text(0.01, 0.02, f'MAE_resid={mae_resid:.4f} | RMSE_resid={rmse_resid:.4f} | Corr={corr_resid:.4f}',
            transform=ax.transAxes, fontsize=8, color='gray', va='bottom')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + f'gambar_{FIG_RESID_HYB[lbl]}_residual_compare_{lbl.lower()}.png',
                 dpi=150, bbox_inches='tight')
    if SHOW_HYBRID_RESIDUAL_PLOTS:
        plt.show()
    else:
        plt.close(fig)

df_resid_compare = pd.DataFrame(resid_compare_rows)
df_resid_compare.to_csv(OUTPUT_DIR + 'tabel_residual_compare_hybrid.csv', index=False)
display(df_resid_compare)


In [ ]:
# Gambar 4.14 — Bar Chart Perbandingan Metrik
model_lbls = df_eval['Model'].tolist()
clrs = [COLORS[m] for m in model_lbls]

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for ax, metric in zip(axes, ['MAE','RMSE','MASE']):
    vals = df_eval[metric].values
    best_idx = int(vals.argmin())
    bars = ax.bar(range(len(model_lbls)), vals, color=clrs, alpha=0.85,
                  edgecolor='white', lw=0.8)
    bars[best_idx].set_edgecolor('gold'); bars[best_idx].set_linewidth(3)
    ax.set_xticks(range(len(model_lbls)))
    ax.set_xticklabels([m.replace('_','-') for m in model_lbls],
                        rotation=45, ha='right', fontsize=7.5)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim(0, max(vals) * 1.18)  # beri ruang ekstra di atas bar
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.03,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold',
                rotation=90 if len(model_lbls) > 8 else 0)
    ax.text(0.5, 0.97, f'Terbaik: {model_lbls[best_idx]}',
            transform=ax.transAxes, ha='center', va='top', fontsize=8, color='goldenrod')

plt.suptitle('Perbandingan Metrik 12 Skenario Model',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'gambar_4_14_barplot_metrik.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Gambar 4.15 — Error Distribution (Boxplot)
fig, ax = plt.subplots(figsize=(16, 6))
err_data, err_labels, err_colors = [], [], []
for lbl in MODEL_ORDER:
    if lbl not in STAGE2:
        continue
    p = STAGE2[lbl]['pred']; a = STAGE2[lbl]['actual']
    n_p = min(len(p), len(a))
    err_data.append(a[-n_p:] - p[-n_p:])
    err_labels.append(lbl.replace('_','-'))
    err_colors.append(COLORS[lbl])

bps = ax.boxplot(err_data, labels=err_labels, patch_artist=True, notch=False,
                  medianprops={'color':'black','lw':2})
for patch, color in zip(bps['boxes'], err_colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.axhline(y=0, color='black', ls='--', lw=1, alpha=0.6)
ax.set_title('Distribusi Error Prediksi per Model',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Error (Aktual - Prediksi)'); ax.set_xlabel('Model')
ax.tick_params(axis='x', labelrotation=35, labelsize=8.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'gambar_4_15_error_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()


## 15. Export Semua Hasil ke CSV

In [ ]:
# CSV 1: Prediksi lengkap vs aktual
pred_dict = {'Date': test.index[-n_ref:].strftime('%Y-%m').tolist(),
             'Aktual': [round(float(v),4) for v in actual]}
for lbl in MODEL_ORDER:
    if lbl not in STAGE2:
        pred_dict[lbl] = [None]*n_ref; continue
    p  = STAGE2[lbl]['pred']
    n_p = len(p)
    pred_dict[lbl] = [None]*(n_ref-n_p) + [round(float(v),4) for v in p]
df_pred_all = pd.DataFrame(pred_dict)
df_pred_all.to_csv(OUTPUT_DIR + 'csv_prediksi_test_stage2.csv', index=False)

# CSV 2: Evaluasi final
df_eval.to_csv(OUTPUT_DIR + 'csv_evaluasi_stage2.csv', index=False)

# CSV 3: Rangkuman Stage 1 SARIMAX order (4 skenario)
sarimax_order_rows = []
for lbl in SARIMAX_SCENARIOS:
    sp = sarimax_specs[lbl]
    wn = STAGE2.get(lbl, {}).get('is_wn', None)
    sarimax_order_rows.append({
        'Model': lbl, 'order': str(sp['order']),
        'sorder': str(sp['sorder']),
        'AIC_best': sp['df_top5']['AIC'].min() if len(sp['df_top5'])>0 else None,
        'White_Noise': wn
    })
pd.DataFrame(sarimax_order_rows).to_csv(OUTPUT_DIR+'csv_sarimax_orders.csv', index=False)

# CSV 4: Residual diagnostik gabungan
df_lb_all.to_csv(OUTPUT_DIR + 'csv_ljungbox_gabungan.csv', index=False)

# CSV 5: Hyperparameter LSTM & Hybrid best (8 skenario)
df_tune.to_csv(OUTPUT_DIR + 'csv_best_hyperparameter_lstm.csv', index=False)

saved_files = sorted(os.listdir(OUTPUT_DIR))
csv_files = [name for name in saved_files if name.endswith(".csv")]
png_files = [name for name in saved_files if name.endswith(".png")]

export_summary = pd.DataFrame({
    "Jenis": ["CSV", "PNG"],
    "Jumlah": [len(csv_files), len(png_files)],
    "Direktori": [OUTPUT_DIR, OUTPUT_DIR],
})
display(export_summary)

if SHOW_EXPORT_FILE_LIST:
    display(pd.DataFrame({"CSV": pd.Series(csv_files), "PNG": pd.Series(png_files)}))


In [ ]:
df_final = df_eval.sort_values("MAE").reset_index(drop=True)
best_model = df_final.iloc[0]["Model"]

print("RINGKASAN AKHIR")
print(f"Periode data : {data_tr.index[0]:%b %Y} – {data_tr.index[-1]:%b %Y}")
print(f"Train final  : {n_train_s2} observasi")
print(f"Test         : {n_test} observasi")
print(f"Model terbaik: {best_model}")

display(df_final[["Model", "MAE", "RMSE", "MASE", "Rank MAE"]])
